# Studi Kasus Sesi 6 — Klasifikasi Penyebab Peringatan Deforestasi
### Rule-Based vs. Machine Learning Konvensional pada Data Simulasi Alert Satelit

> **Catatan:** seluruh data yang dipakai pada notebook ini adalah **data
> simulasi (sintetis)**, dibuat secara terprogram dengan seed tetap agar
> hasilnya selalu reproducible.

**Sumber materi:** Webinar *"Deforestation Alerts and Driver Classification"*
oleh **Dr. Bart Slagter** (Wageningen University & Research).

## Daftar Isi Notebook

| Bagian | Isi |
|---|---|
| 0 | Pengantar, catatan data simulasi, pustaka yang dibutuhkan |
| 1 | Pembuatan data dummy (1.500 alert, seed tetap, 7 kelas penyebab) |
| 2 | EDA: distribusi kelas, seasonality per bulan, korelasi fitur |
| 3 | Baseline: klasifikasi **Rule-Based** (if–else) |
| 4 | **Decision Tree** & **Random Forest** (train/test split 80/20 stratified) |
| 5 | Evaluasi: accuracy, precision, recall, F1, confusion matrix, tabel perbandingan |
| 6 | Feature importance (Random Forest) & visualisasi pohon keputusan |
| 7 | Analisis kesalahan: pasangan kelas yang tertukar & kaitannya dengan materi webinar |
| 8 | Simulasi pembaruan bulanan (uji latensi/operasional seperti sistem peringatan nyata) |
| 9 | Kerangka laporan (Latar Belakang → Kesimpulan) |

**Alur logis yang mengikuti rekomendasi pemateri:** *mulailah dari metode
konvensional sederhana sebagai benchmark, baru pertimbangkan deep learning
hanya bila masalahnya memang membutuhkannya.*

## 0. Persiapan Lingkungan

Sel ini siapkan pustaka, **seed acak** (agar hasil selalu sama), lokasi
penyimpanan file, dan gaya visual grafik.

- Di **Google Colab**: file disimpan ke `/content`.
- Di **laptop**: file disimpan ke folder `data/` di samping notebook.

**Pustaka yang dibutuhkan:** `pandas`, `numpy`, `matplotlib`, `seaborn`,
`scikit-learn`. Semuanya sudah tersedia bawaan di Google Colab. Untuk
instalasi manual: `pip install pandas numpy matplotlib seaborn scikit-learn`.

In [ ]:
# ============================================================
# 0. PERSIAPAN LINGKUNGAN
# ============================================================
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# --- Seed global: memastikan SELURUH proses acak reproducible ---
SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# --- Lokasi output: otomatis membaca environment Colab vs lokal ---
BASE_DIR = "/content" if os.path.isdir("/content") else str(Path.cwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
FIG_DIR = os.path.join(DATA_DIR, "figures")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
CSV_ALERT = os.path.join(DATA_DIR, "alert_deforestation_simulasi.csv")

# --- Gaya visual yang konsisten untuk seluruh grafik ---
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"


def simpan_fig(nama_file, dpi=110):
    # Simpan grafik yang sedang aktif ke folder figures, lalu tampilkan di notebook.
    try:
        plt.savefig(os.path.join(FIG_DIR, nama_file), dpi=dpi, bbox_inches="tight")
    except Exception as galat:
        print("Gagal menyimpan gambar:", galat)
    plt.show()


print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("scikit-learn:", __import__("sklearn").__version__)
print("Lokasi data :", DATA_DIR)
print("Seed global :", SEED)

## 1. Pembuatan Data Dummy (Sintetis) — 1.500 Peringatan Deforestasi

### 1.1 Konsep data

Satu baris = **satu *alert* deforestasi** (satu poligon/piksel 10–30 m yang
dihasilkan secara otomatis oleh sistem peringatan, misalnya GLAD, RADD,
atau TreeCoverer).
Alert hanya memberi tahu **DI MANA** tutupan pohon hilang — **bukan** penyebabnya.
Tugas machine learning di sini adalah **mengklasifikasikan *direct driver***
(penyebab langsung), sebagaimana yang dilakukan pemateri dengan CNN.

### 1.2 Definisi kolom

| Kolom | Tipe | Makna |
|---|---|---|
| `luas_ha` | float | luas area alert (hektare) |
| `ada_titik_api` | int (0/1) | apakah ada *hotspot* aktif dari sensor api (FIRMS/VIIRS) yang bertepatan dengan alert |
| `jarak_ke_jalan_km` | float | jarak alert ke jalan/terraform terdekat |
| `jarak_ke_sungai_km` | float | jarak alert ke sungai/danau terdekat |
| `bulan` | int (1–12) | bulan saat alert dipublikasikan |
| `bentuk_memanjang` | float (0–1) | rasio bentuk area (tinggi = memanjang seperti jalan) |
| `kedekatan_tambang_km` | float | jarak ke konsesi tambang terdekat |
| `label_penyebab` | kategori | **target**: 7 penyebab langsung |

### 1.3 Logika pola yang disimulasikan (mengikuti materi webinar)

| Kelas | Pola utama yang disimulasikan |
|---|---|
| `pertanian_kecil` | luas kecil, bentuk kompak (petak kecil), dekat jalan, pembakaran relatif sering |
| `pertanian_besar` | luas **besar** (puluhan–ratusan ha), Agro-industrial, bentuk persegi, sering di musim kemarau |
| `selective_logging` | luas sedang-kecil, **sangat dekat jalan** (jalan akses logging), bentuk agak memanjang |
| `jalan` | luas kecil tapi **sangat memanjang** (bentuk memanjang tinggi), jarak ke jalan ~0 |
| `tambang` | **sangat dekat konsesi tambang**, bentuk kompak, area pembukaan lahan |
| `kebakaran` | **ada titik api**, luas besar, **terpusat di bulan kemarau (6–9)**, sering di tepi sungai |
| `alami` | **sangat dekat sungai** (banjir), **bermusim hujan**, kadang muncul titik api (banjir → sinyal panas/panas) |

### 1.4 Noise label (penting!)

Sekitar **12% label** diberi noise: satu baris di-*relabel* ke kelas yang
*mirip/tertukar* (mis. `kebakaran` → `alami`). Ini sengaja dilakukan untuk
meniru kondisi nyata:

1. **Kebakaran vs banjir** sulit dibedakan dari citra (temuan pemateri).
2. **Lahan terbakar** sering kemudian berubah menjadi lahan pertanian (peralihan
   penggunaan lahan).
3. Ground truth labeling di lapangan selalu punya **ketidakpastian**.

Efeknya: model **tidak mungkin** mencapai akurasi 100% — akan ada *error floor*
sekitar 12% yang mustahil diatasi. Ini justru pesan penting dari webinar:
**prediksi AI bukan kebenaran mutlak.**

In [ ]:
# ============================================================
# 1. PEMBUATAN DATA SIMULASI
# ============================================================
KELAS = [
    "pertanian_kecil",
    "pertanian_besar",
    "selective_logging",
    "jalan",
    "tambang",
    "kebakaran",
    "alami",
]

# Urutan label yang DIURUTKAN (sklearn mengurutkan label kelas secara alfabetis
# di dalam model, jadi nama kelas untuk visualisasi pohon harus ikut terurut)
KELAS_URUT = sorted(KELAS)

# Proporsi tiap kelas (tidak seimbang total: cukup realistis untuk deforestation alert)
PEMBAGIAN = np.array([0.22, 0.13, 0.18, 0.12, 0.12, 0.13, 0.10])

# Bobot musim (Jan..Des) per kelas -> mencerminkan seasonality seperti di webinar
BOBOT_BULAN = {
    #            Jan Feb Mar Apr Mei Jun Jul Agu Sep Okt Nov Des
    "pertanian_kecil": [1.0, 1.0, 1.2, 1.2, 1.0, 0.9, 0.8, 0.8, 1.0, 1.2, 1.3, 1.2],
    "pertanian_besar": [0.8, 0.8, 1.0, 1.2, 1.5, 2.0, 2.0, 1.8, 1.5, 1.2, 1.0, 0.8],
    "selective_logging": [1.0] * 12,
    "jalan":          [1.0] * 12,
    "tambang":       [1.0] * 12,
    # Kebakaran: terkonsentrasi kuat di musim kemarau
    "kebakaran":      [0.3, 0.3, 0.5, 0.8, 1.5, 3.0, 4.0, 4.0, 3.0, 1.5, 0.6, 0.3],
    # Alami (banjir): sebaliknya, terkonsentrasi di musim hujan
    "alami":         [2.0, 2.0, 1.8, 1.5, 1.0, 0.8, 0.7, 0.7, 0.8, 1.0, 1.8, 2.0],
}

# Tetangga kelas untuk noise label (pasangan yang memang sering tertukar di lapangan)
TETANGGA_NOISE = {
    "pertanian_kecil": ["pertanian_besar", "selective_logging", "alami"],
    "pertanian_besar": ["pertanian_kecil", "tambang", "selective_logging"],
    "selective_logging": ["jalan", "pertanian_kecil", "tambang"],
    "jalan":          ["selective_logging", "pertanian_kecil"],
    "tambang":       ["selective_logging", "pertanian_besar"],
    # Kebakaran dan banjir adalah pasangan kelas yang paling sering tertukar
    "kebakaran":      ["alami", "pertanian_kecil", "pertanian_besar"],
    "alami":         ["kebakaran", "pertanian_kecil"],
}

TINGKAT_NOISE = 0.12  # 12% label diberi noise (ground truth yang ambigu)


def _bulan(rng, kelas):
    # Ambil bulan (1-12) mengikuti bobot musim kelas tersebut.
    bobot = np.array(BOBOT_BULAN[kelas], dtype=float)
    bobot = bobot / bobot.sum()
    return int(rng.choice(np.arange(1, 13), p=bobot))


def _sampel_kelas(rng, kelas):
    # Buat 1 baris fitur yang mengikuti pola khas kelas tersebut.
    baris = {}

    # --- 1) LUAS AREA (distribusi miring: banyak yang kecil, sedikit yang besar) ---
    profil_luas = {
        "pertanian_kecil":   lambda: rng.gamma(2.0, 1.0) + 0.2,     # median ~1,7 ha
        "pertanian_besar":   lambda: rng.gamma(2.5, 55.0) + 20.0,   # median ~140 ha
        "selective_logging": lambda: rng.gamma(1.8, 3.0) + 0.4,     # median ~5 ha
        "jalan":            lambda: rng.gamma(1.4, 1.0) + 0.3,     # median ~1,3 ha
        "tambang":         lambda: rng.gamma(2.0, 6.0) + 1.5,      # median ~13 ha
        "kebakaran":        lambda: rng.gamma(2.2, 22.0) + 2.0,     # median ~45 ha
        "alami":           lambda: rng.gamma(2.0, 9.0) + 1.5,      # median ~19 ha
    }
    baris["luas_ha"] = profil_luas[kelas]()

    # --- 2) TITIK API (hotspot) ---
    if kelas == "kebakaran":
        baris["ada_titik_api"] = int(rng.random() < 0.90)   # 10% wildfire tanpa hotspot terdeteksi
    elif kelas == "alami":
        baris["ada_titik_api"] = int(rng.random() < 0.30)   # lahan basah/banjir ikut terbaca sebagai "panas"
    else:
        baris["ada_titik_api"] = int(rng.random() < 0.03)   # false positive kecil

    # --- 3) JARAK KE JALAN ---
    # lognormal dipakai agar jarak selalu positif dan ber-distribusi panjang ("jarak jauh")
    profil_jalan = {
        "jalan":            lambda: rng.uniform(0.0, 0.40),        # alert itu sendiri ada di jalan
        "selective_logging": lambda: rng.gamma(1.3, 0.7) + 0.05,   # jalan akses logging
        "pertanian_besar":  lambda: rng.lognormal(0.9, 0.6),        # median ~2,5 km
        "pertanian_kecil":  lambda: rng.lognormal(1.6, 0.7),        # median ~5 km
        "tambang":         lambda: rng.lognormal(1.5, 0.8),        # median ~4,5 km
        "kebakaran":       lambda: rng.lognormal(2.2, 0.9),        # median ~9 km
        "alami":           lambda: rng.lognormal(2.0, 0.9),        # median ~7,4 km
    }
    baris["jarak_ke_jalan_km"] = profil_jalan[kelas]()

    # --- 4) JARAK KE SUNGAI ---
    profil_sungai = {
        "alami":           lambda: rng.gamma(1.1, 0.6) + 0.05,     # di dataran banjir (floodplain)
        "kebakaran":       lambda: rng.gamma(1.6, 2.2) + 0.20,     # kebakaran tepi sungai
        "jalan":           lambda: rng.lognormal(1.6, 0.8),
        "selective_logging": lambda: rng.lognormal(2.0, 0.8),
        "pertanian_besar": lambda: rng.lognormal(2.1, 0.8),
        "pertanian_kecil": lambda: rng.lognormal(2.3, 0.9),
        "tambang":         lambda: rng.lognormal(2.2, 0.9),
    }
    baris["jarak_ke_sungai_km"] = profil_sungai[kelas]()

    # --- 5) KEDEKATAN TAMBANG ---
    if kelas == "tambang":
        baris["kedekatan_tambang_km"] = rng.uniform(0.0, 2.5)       # di dalam konsesi tambang
    elif kelas == "selective_logging":
        baris["kedekatan_tambang_km"] = rng.lognormal(1.9, 0.7)     # konsesi logging
    else:
        baris["kedekatan_tambang_km"] = rng.uniform(12.0, 60.0)     # jauh dari tambang

    # --- 6) BENTUK MEMANJANG (0 = kompak, 1 = memanjang/garis) ---
    if kelas == "jalan":
        baris["bentuk_memanjang"] = rng.beta(8.0, 1.5)
    elif kelas == "selective_logging":
        baris["bentuk_memanjang"] = rng.beta(3.0, 4.5)   # agak memanjang, tapi tidak seperti jalan
    elif kelas == "kebakaran":
        baris["bentuk_memanjang"] = rng.beta(2.2, 3.0)   # bentuk api tidak beraturan
    elif kelas == "alami":
        baris["bentuk_memanjang"] = rng.beta(2.0, 3.0)
    elif kelas == "pertanian_besar":
        baris["bentuk_memanjang"] = rng.beta(1.3, 5.0)   # petak persegi, relatif kotak
    else:
        baris["bentuk_memanjang"] = rng.beta(1.5, 5.0)

    # --- 7) BULAN ---
    baris["bulan"] = _bulan(rng, kelas)
    return baris


def buat_data(n_baris=1500, seed=42):
    # Membuat dataset alert deforestasi sintetis yang reproducible.
    r = np.random.default_rng(seed)
    n_per_kelas = (r.multinomial(n_baris, PEMBAGIAN / PEMBAGIAN.sum()))
    baris = []
    for kelas, jml in zip(KELAS, n_per_kelas):
        for _ in range(int(jml)):
            b = _sampel_kelas(r, kelas)
            b["label_penyebab"] = kelas
            baris.append(b)

    df = pd.DataFrame(baris)

    # --- Noise pengukuran: tambah derau kecil pada fitur numerik ---
    df["luas_ha"] += r.normal(0, df["luas_ha"].std() * 0.05, len(df))
    df["jarak_ke_jalan_km"] += r.normal(0, 0.25, len(df))
    df["jarak_ke_sungai_km"] += r.normal(0, 0.30, len(df))
    df["kedekatan_tambang_km"] += r.normal(0, 1.0, len(df))
    df["bentuk_memanjang"] += r.normal(0, 0.03, len(df))

    # --- Noise label: 12% baris di-relabel ke kelas yang mirip/sulit dibedakan ---
    indeks_noise = r.choice(len(df), size=int(len(df) * TINGKAT_NOISE), replace=False)
    for i in indeks_noise:
        lama = df.at[i, "label_penyebab"]
        kandidat = TETANGGA_NOISE[lama]
        df.at[i, "label_penyebab"] = kandidat[r.integers(len(kandidat))]

    # --- Bersihkan & impose rentang fisik yang masuk akal ---
    df["luas_ha"] = df["luas_ha"].clip(lower=0.05).round(2)
    df["jarak_ke_jalan_km"] = df["jarak_ke_jalan_km"].clip(lower=0.0).round(2)
    df["jarak_ke_sungai_km"] = df["jarak_ke_sungai_km"].clip(lower=0.0).round(2)
    df["kedekatan_tambang_km"] = df["kedekatan_tambang_km"].clip(lower=0.0).round(2)
    df["bentuk_memanjang"] = df["bentuk_memanjang"].clip(0.0, 1.0).round(3)
    df["ada_titik_api"] = df["ada_titik_api"].astype(int)
    df["bulan"] = df["bulan"].astype(int)

    # Urutkan kolom agar rapi
    kolom = [
        "luas_ha", "ada_titik_api", "jarak_ke_jalan_km", "jarak_ke_sungai_km",
        "bulan", "bentuk_memanjang", "kedekatan_tambang_km", "label_penyebab",
    ]
    return df[kolom].sample(frac=1.0, random_state=seed).reset_index(drop=True)


# --- Jalankan pembuatan dataset ---
df = buat_data(n_baris=1500, seed=SEED)
df.to_csv(CSV_ALERT, index=False, encoding="utf-8")

print(f"Jumlah baris : {len(df)}")
print(f"Jumlah kolom : {df.shape[1]}")
print(f"File CSV     : {CSV_ALERT}")
print("\nContoh 5 baris pertama:")
print(df.head().to_string(index=False))

In [ ]:
# Cek cepat: apakah pola yang disimulasikan memang terbentuk? (sanity check)
ringkas = (
    df.groupby("label_penyebab")
    .agg(
        n=("luas_ha", "size"),
        luas_median=("luas_ha", "median"),
        titik_api=("ada_titik_api", "mean"),
        d_jalan_med=("jarak_ke_jalan_km", "median"),
        d_sungai_med=("jarak_ke_sungai_km", "median"),
        memanjang=("bentuk_memanjang", "mean"),
        d_tambang_med=("kedekatan_tambang_km", "median"),
    )
    .round(2)
)
print(ringkas.to_string())

## 2. Exploratory Data Analysis (EDA)

Tiga pertanyaan yang dijawab:

1. **Berapa banyak alert per kelas penyebab?** ( apakah dataset seimbang?)
2. **Kapan (bulan mana) tiap penyebab terjadi?** — ini menguji temuan webinar
   tentang *seasonality*: kebakaran terkonsentrasi di musim kemarau, banjir di
   musim hujan, dan nilai *seasonality* yang tinggi di Amazon/Basin Congo
   dibanding Asia Tenggara.
3. **Bagaimana korelasi antar fitur?** — untuk melihat fitur mana yang
   saling redundant (bisa menyebabkan *multicollinearity* pada model linear).

In [ ]:
# ============================================================
# 2.1 DISTRIBUSI KELAS
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5))
urutan = df["label_penyebab"].value_counts()
warna = sns.color_palette("viridis", len(urutan))
bars = ax.bar(urutan.index, urutan.values, color=warna, edgecolor="black", linewidth=0.6)
ax.set_xlabel("Penyebab langsung (kelas target)")
ax.set_ylabel("Jumlah alert (n)")
ax.set_xticklabels(urutan.index, rotation=25, ha="right")
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 8,
            f"{int(b.get_height())}\n({b.get_height()/len(df)*100:.1f}%)",
            ha="center", va="bottom", fontsize=9)
ax.set_title("Distribusi Alert Deforestasi per Penyebab Langsung\n"
             "(data simulasi/sintetis, seed=42)", fontsize=13, weight="bold")
simpan_fig("01_distribusi_kelas.png")

In [ ]:
# ============================================================
# 2.2 SEASONALITY: JUMLAH ALERT PER BULAN PER PENYEBAB
# ============================================================
tabel_bulan = pd.crosstab(df["bulan"], df["label_penyebab"])

fig, ax = plt.subplots(figsize=(13, 6))
tabel_bulan.plot(kind="bar", stacked=True, ax=ax, colormap="Set2",
                 edgecolor="white", linewidth=0.5)
ax.set_title("Seasonality Alert Deforestasi per Bulan (2024, 1 tahun penuh)", fontsize=13, weight="bold")
ax.set_xlabel("Bulan (1 = Januari, 12 = Desember)")
ax.set_ylabel("Jumlah alert (n)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "Mei", "Jun",
                    "Jul", "Agu", "Sep", "Okt", "Nov", "Des"])
ax.legend(title="Penyebab", ncol=4, fontsize=9)
simpan_fig("02_seasonality_bulanan_stacked.png")

In [ ]:
# Version "garis" per kelas -> lebih mudah membaca pola musim tiap penyebab
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
for ax, kelas in zip(axes.flat, KELAS):
    if kelas not in tabel_bulan.columns:
        ax.axis("off")
        continue
    ax.plot(tabel_bulan.index, tabel_bulan[kelas], marker="o", color="#C0392B", linewidth=2)
    ax.axvspan(6, 9, color="orange", alpha=0.15)   # Perhatikan: bulan 6-9 = musim kemarau
    ax.set_title(kelas, fontsize=11, weight="bold")
    ax.set_xlabel("Bulan")
    ax.set_ylabel("Jumlah alert")
    ax.set_xticks(range(1, 13, 2))
axes.flat[-1].axis("off")
fig.suptitle("Pola Musiman Tiap Penyebab Deforestasi (area oranye = musim kemarau, Jun–Sep)",
             fontsize=13, weight="bold", y=1.02)
plt.tight_layout()
simpan_fig("03_seasonality_per_kelas.png")

In [ ]:
# ============================================================
# 2.3 KORELASI FITUR + RINGKASAN STATISTIK
# ============================================================
fitur_num = ["luas_ha", "ada_titik_api", "jarak_ke_jalan_km",
             "jarak_ke_sungai_km", "bulan", "bentuk_memanjang",
             "kedekatan_tambang_km"]

matriks_kor = df[fitur_num].corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(matriks_kor, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Matriks Korelasi Spearman Antarfitur Numerik", fontsize=13, weight="bold")
ax.set_xlabel("Fitur")
ax.set_ylabel("Fitur")
simpan_fig("04_korelasi_fitur.png")

print("Ringkasan statistik fitur:")
print(df[fitur_num].describe().round(2).to_string())

## 3. Baseline: Klasifikasi **Rule-Based** (if–else)

Sebelum memakai machine learning, cara paling sederhana (dan paling
"terjelaskan") adalah menuliskan **aturan(if–else)** secara manual — persis
seperti yang dilakukan analis.remote sensing pada dekade sebelum *deep learning*
mendarat.

Aturan disusun mengikuti logika geografis (angka ambangnya bisa dibaca langsung
pada kode sel berikutnya):

1. Sangat memanjang + luas < 8 ha + nyaris menempel di jalan → **jalan**
2. Sangat dekat konsesi tambang (< 2,5 km) → **tambang**
3. Ada titik api + bulan kemarau (Jun–Sep) → **kebakaran**
4. Sangat dekat sungai (< 1,2 km) → **alami** (banjir)
5. Ada titik api (di luar musim kemarau) → **kebakaran**
6. Dekat jalan akses (< 0,9 km) + luas kecil + tidak memanjang → **selective_logging**
7. Luas ≥ 30 ha → **pertanian_besar**
8. Selain itu → **pertanian_kecil**

> **Catatan:** urutan `if` sangat memengaruhi. Perhatikan bahwa aturan
> *kebakaran* diletakkan **sebelum** *alami* — inilah yang menyebabkan
> sebagian besar error pada model ini dan menjadi bahan bahasan di Bagian 7.
> Aturan *tambang* juga diletakkan sebelum *selective_logging*, padahal
> konsesi logging sering berada dekat konsesi tambang — konsekuensinya kelas
> `selective_logging` hanya dikenali pada sebagian kecil alert.

In [ ]:
# ============================================================
# 3. KLASIFIKASI RULE-BASED
# ============================================================
MUSIM_KEMARAU = [6, 7, 8, 9]


def klasifikasi_rule_based(row):
    # Klasifikasi 1 baris alert berdasarkan aturan geografis/seasonal.
    # 1) Jalan: bentuk sangat memanjang, luas kecil, dan nyaris menempel di jalan
    if row["bentuk_memanjang"] > 0.60 and row["luas_ha"] < 8 and row["jarak_ke_jalan_km"] < 0.8:
        return "jalan"
    # 2) Tambang: sangat dekat konsesi tambang
    if row["kedekatan_tambang_km"] < 2.5:
        return "tambang"
    # 3) Kebakaran: ada hotspot DAN terjadi di musim kemarau
    if row["ada_titik_api"] == 1 and row["bulan"] in MUSIM_KEMARAU:
        return "kebakaran"
    # 4) Alami (banjir): sangat dekat sungai
    if row["jarak_ke_sungai_km"] < 1.2:
        return "alami"
    # 5) Kebakaran: ada hotspot di luar musim kemarau
    if row["ada_titik_api"] == 1:
        return "kebakaran"
    # 6) Selective logging: dekat jalan akses, luas kecil, bentuk tidak terlalu memanjang
    if row["jarak_ke_jalan_km"] < 0.9 and row["luas_ha"] < 25 and row["bentuk_memanjang"] < 0.60:
        return "selective_logging"
    # 7) Pertanian besar: luas sangat besar
    if row["luas_ha"] >= 30:
        return "pertanian_besar"
    # 8) Default: pertanian kecil
    return "pertanian_kecil"


# Terapkan aturan ke seluruh dataset
df["prediksi_rule_based"] = df.apply(klasifikasi_rule_based, axis=1)

akurasi_rule = accuracy_score(df["label_penyebab"], df["prediksi_rule_based"])
print(f"Akurasi rule-based pada SELURUH data : {akurasi_rule:.4f}  ({akurasi_rule*100:.2f}%)")
print(f"Jumlah alert                         : {len(df)}")
print(f"Jumlah salah klasifikasi            : {int((df['label_penyebab'] != df['prediksi_rule_based']).sum())}")
print("\nConfusion matrix rule-based (baris =-label asli, kolom = prediksi):")
print(pd.crosstab(df["label_penyebab"], df["prediksi_rule_based"]).to_string())

## 4. Decision Tree & Random Forest

### 4.1 Mengapa bukan langsung deep learning?

Pemateri sendiri menyatakan: *"selalu baik untuk mencoba metode machine
learning konvensional atau bahkan metode yang lebih sederhana dulu; bila
masalahnya menjadi sangat kompleks — misalnya harus membaca konteks, atau
time series yang kompleks — barulah deep learning diperlukan."*

Di sini masalahnya **sederhana**: 7 fitur tabuler → 7 kelas. Decision Tree
dan Random Forest adalah cara paling logis untuk menguji apakah data
cukup informatif. Deep learning baru masuk akal bila:
- masukannya berupa **stack citra multi-sensor** (Sentinel-1 + Sentinel-2),
- atau data berlabel-nya sangat banyak, atau
- polanya yang dicari bersifat **spasial-temporal** (pola berulang antartahun).

### 4.2 Konfigurasi eksperimen

- Split **80/20 stratified** → menjaga proporsi kelas di train & test.
- **Decision Tree**: `max_depth=8`, `min_samples_leaf=5`, `class_weight='balanced'`
  (kedalaman dibatasi agar model masih bisa dibaca & tidak overfit).
- **Random Forest**: `n_estimators=400`, `min_samples_leaf=2`,
  `class_weight='balanced_subsample'`.
- Validasi tambahan: **5-fold cross-validation** untuk mengukur kestabilan.

In [ ]:
# ============================================================
# 4. PELATIHAN MODEL
# ============================================================
FITUR = ["luas_ha", "ada_titik_api", "jarak_ke_jalan_km",
         "jarak_ke_sungai_km", "bulan", "bentuk_memanjang",
         "kedekatan_tambang_km"]
TARGET = "label_penyebab"

X = df[FITUR]
y = df[TARGET]

# --- Split 80/20 stratified ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f"Jumlah data latih : {len(X_train)}")
print(f"Jumlah data uji   : {len(X_test)}")
print(f"Proporsi kelas di train set:")
print((y_train.value_counts(normalize=True) * 100).round(1).to_string())
print(f"\nProporsi kelas di test set:")
print((y_test.value_counts(normalize=True) * 100).round(1).to_string())

In [ ]:
# --- Model 1: Decision Tree ---
dt = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=SEED,
)

# --- Model 2: Random Forest ---
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1,
)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

pred_dt = dt.predict(X_test)
pred_rf = rf.predict(X_test)

print("=== Decision Tree ===")
print(f"Akurasi  : {accuracy_score(y_test, pred_dt):.4f}")
print(f"Macro F1 : {f1_score(y_test, pred_dt, average='macro'):.4f}")
print("\n=== Random Forest ===")
print(f"Akurasi  : {accuracy_score(y_test, pred_rf):.4f}")
print(f"Macro F1 : {f1_score(y_test, pred_rf, average='macro'):.4f}")

In [ ]:
# --- Validasi silang 5-fold: seberapa stabil hasilnya? ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
skor_dt = cross_val_score(dt, X, y, cv=cv, scoring="accuracy")
skor_rf = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")

print("=== 5-Fold Cross-Validation (akurasi) ===")
print(f"Decision Tree : {skor_dt.mean():.4f} (std {skor_dt.std():.4f}) -> {skor_dt.round(4)}")
print(f"Random Forest : {skor_rf.mean():.4f} (std {skor_rf.std():.4f}) -> {skor_rf.round(4)}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.boxplot([skor_dt, skor_rf], labels=["Decision Tree", "Random Forest"])
ax.set_title("Stabilitas Akurasi 5-Fold Cross-Validation", fontsize=13, weight="bold")
ax.set_xlabel("Model")
ax.set_ylabel("Akurasi (meanCV)")
ax.set_ylim(min(skor_dt.min(), skor_rf.min()) - 0.05, 1.0)
simpan_fig("05_cross_validation.png")

## 5. Evaluasi Tiga Model (Rule-Based, Decision Tree, Random Forest)

Metrik yang dihitung:

- **Accuracy** — proporsi prediksi benar secara keseluruhan.
- **Precision** — dari semua yang *diprediksi* sebagai kelas X, berapa yang benar?
  (penting untuk situasi lapangan: mengurangi *false alarm* dalam penegakan hukum).
- **Recall** — dari semua yang *SEBENARNYA* kelas X, berapa yang berhasil ditemukan?
  (penting agar tidak ada deforestation yang lolos).
- **F1-score** — harmonic mean dari precision & recall.

> **Catatan penting (align dengan materi webinar):** untuk penegakan hukum
> (law enforcement), kedua metrik sama penting. Presisi rendah = banyak
> *false alarm* yang membuang sumber daya. Recall rendah = deforestasi ilegal
> yang lolos dari radar.

In [ ]:
# ============================================================
# 5.1 TABEL PERBANDINGAN KETIGA MODEL
# ============================================================
def ringkasan_model(nama, y_true, y_pred):
    # Kembalikan baris ringkasan metrik untuk satu model.
    return {
        "Model": nama,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


pred_rule_test = df.loc[X_test.index, "prediksi_rule_based"].values

tabel_perbandingan = pd.DataFrame([
    ringkasan_model("Rule-Based (if-else)", y_test, pred_rule_test),
    ringkasan_model("Decision Tree", y_test, pred_dt),
    ringkasan_model("Random Forest", y_test, pred_rf),
]).set_index("Model").round(4)

print("=== TABEL PERBANDINGAN MODEL (data uji, n = %d) ===" % len(X_test))
print(tabel_perbandingan.to_string())

In [ ]:
# ============================================================
# 5.2 METRIK PER KELAS
# ============================================================
def metrik_per_kelas(y_true, y_pred):
    # Precision/Recall/F1/support per kelas.
    lap = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    baris = []
    for kelas in KELAS:
        baris.append({
            "Kelas": kelas,
            "Precision": lap[kelas]["precision"],
            "Recall": lap[kelas]["recall"],
            "F1": lap[kelas]["f1-score"],
            "Support": int(lap[kelas]["support"]),
        })
    return pd.DataFrame(baris).set_index("Kelas")


m_rule = metrik_per_kelas(y_test, pred_rule_test)
m_dt = metrik_per_kelas(y_test, pred_dt)
m_rf = metrik_per_kelas(y_test, pred_rf)

for judul, tabel in [("RULE-BASED", m_rule), ("DECISION TREE", m_dt), ("RANDOM FOREST", m_rf)]:
    print(f"\n=== {judul} — metrik per kelas ===")
    print(tabel.round(3).to_string())

In [ ]:
# ============================================================
# 5.3 VISUALISASI PERBANDINGAN PRECISION & RECALL ANTAR KELAS
# ============================================================
# Ubah tabel metrik per model ke format "panjang" agar mudah diplot
bagian_long = []
for nama_model, tabel in [("Rule-Based", m_rule), ("Decision Tree", m_dt), ("Random Forest", m_rf)]:
    t = tabel.reset_index().melt(id_vars="Kelas", var_name="Metrik", value_name="Nilai")
    t["Model"] = nama_model
    bagian_long.append(t)
data_long = pd.concat(bagian_long, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
x = np.arange(len(KELAS))
lebar = 0.26
warna_model = {"Rule-Based": "#8D99AE", "Decision Tree": "#F4A261", "Random Forest": "#2A9D8F"}

for ax, metrik in zip(axes, ["Precision", "Recall"]):
    subset = data_long[data_long["Metrik"] == metrik]
    for i, (nama_model, grup) in enumerate(subset.groupby("Model", sort=False)):
        grup = grup.set_index("Kelas").reindex(KELAS)
        ax.bar(x + (i - 1) * lebar, grup["Nilai"], lebar,
               label=nama_model, color=warna_model[nama_model])
    ax.set_title(f"{metrik} per Penyebab Deforestasi", fontsize=13, weight="bold")
    ax.set_xlabel("Kelas penyebab")
    ax.set_ylabel(metrik)
    ax.set_xticks(x)
    ax.set_xticklabels(KELAS, rotation=25, ha="right")
    ax.set_ylim(0, 1.05)
    ax.legend(title="Model")
plt.tight_layout()
simpan_fig("06_perbandingan_precision_recall.png")

In [ ]:
# ============================================================
# 5.4 CONFUSION MATRIX (HEATMAP) UNTUK KETIGA MODEL
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
model_data = [
    ("Rule-Based (if-else)", pred_rule_test, "#8D99AE"),
    ("Decision Tree", pred_dt, "#F4A261"),
    ("Random Forest", pred_rf, "#2A9D8F"),
]

for ax, (nama, pred, warna) in zip(axes, model_data):
    cm = confusion_matrix(y_test, pred, labels=KELAS)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="YlGnBu",
        xticklabels=KELAS, yticklabels=KELAS, ax=ax,
        linewidths=0.5, linecolor="white", cbar_kws={"label": "jumlah alert"},
    )
    ax.set_title(f"{nama}\n(akurasi = {accuracy_score(y_test, pred):.3f})",
                 fontsize=12, weight="bold")
    ax.set_xlabel("Prediksi model")
    ax.set_ylabel("Kenyataan (label asli)")
    ax.tick_params(axis="x", rotation=40)
    ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
simpan_fig("07_confusion_matrix_tiga_model.png")

## 6. Feature Importance & Visualisasi Pohon Keputusan

### 6.1 Mengapa penting?

Dari webinar: model CNN pemateri tidak hanya melihat **ukuran & bentuk** alert,
tetapi juga **konteks di sekitarnya** (mis. ada jalan akses, ada tanda-jejak
pertanian di dekatnya). Feature importance adalah cara kita menguji "informasi
apa yang benar-benar dipakai model" pada pendekatan tabular.

Dua metrik yang dipakai:

- **Gini importance** (RF bawaan) — cepat, tapi cenderung Bias ke fitur berkardinalitas tinggi.
- **Permutation importance** — diacak satu per satu di data uji; drop akurasi
  menunjukkan seberapa penting fitur tersebut. Lebih tepercaya.

### 6.2 Pembatasan kedalaman pohon

Pohon keputusan di-*plot* dengan `max_depth=4` (dari model yang dilatih dengan
`max_depth=8`) supaya **masih terbaca manusia** — komponen kunci dari
*interpretable AI* yang jadi nilai jual utama pendekatan rule-based/pohon.

In [ ]:
# ============================================================
# 6.1 FEATURE IMPORTANCE (GINI + PERMUTATION)
# ============================================================
gini = pd.Series(rf.feature_importances_, index=FITUR).sort_values(ascending=False)

perm = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=SEED, scoring="accuracy", n_jobs=-1
)
perm_series = pd.Series(perm.importances_mean, index=FITUR).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
gini.sort_values().plot(kind="barh", ax=axes[0], color="#2A9D8F")
axes[0].set_title("Gini Importance (Random Forest)", fontsize=13, weight="bold")
axes[0].set_xlabel("Gini importance")
axes[0].set_ylabel("Fitur")

perm_series.sort_values().plot(kind="barh", ax=axes[1], color="#E76F51")
axes[1].set_title("Permutation Importance (drop akurasi di data uji)", fontsize=13, weight="bold")
axes[1].set_xlabel("Penurunan akurasi (mean, 10x pengacakan)")
axes[1].set_ylabel("Fitur")
plt.tight_layout()
simpan_fig("08_feature_importance.png")

print("=== Urutan(feature importance) ===")
tabel_importance = pd.DataFrame({
    "Gini importance": gini.round(4),
    "Permutation importance": perm_series.round(4),
    "Std (permutasi)": pd.Series(perm.importances_std, index=FITUR).round(4),
}).sort_values("Permutation importance", ascending=False)
print(tabel_importance.to_string())

In [ ]:
# ============================================================
# 6.2 VISUALISASI POHON KEPUTUSAN (kedalaman dibatasi = 4)
# ============================================================
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt,
    feature_names=FITUR,
    class_names=KELAS_URUT,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=4,
    impurity=False,   # sembunyikan baris "value = [...]" agar pohon lebih ringkas
    ax=ax,
)
ax.set_title("Pohon Keputusan (batasi kedalaman 4 agar dapat dibaca)", fontsize=13, weight="bold")
ax.set_xlabel("Aturan.split pada fitur")
ax.set_ylabel("Kedalaman pohon")
simpan_fig("09_pohon_keputusan.png")

# --- Teks aturan yang dihasilkan pohon (versi mudah disalin ke laporan) ---
from sklearn.tree import export_text
print("=== Aturan pohon (potongan, kedalaman 3) ===")
print(export_text(dt, feature_names=FITUR, class_names=KELAS_URUT, max_depth=3))

## 7. Analisis Kesalahan (Error Analysis)

Bagian ini adalah inti dari belajar: **model akurasinya tinggi tidak berarti
modelnya berguna**. Yang penting adalah *di mana* model gagal dan *mengapa*.

Tiga pertanyaan:

1. **Pasangan kelas apa yang paling sering tertukar?** (analisis confusion matrix)
2. **Apakah error konsisten dengan temuan pemateri?** —ahi pemateri secara
   eksplisit menyebut: *"kami melihat banyak kesalahan, terutama pada klasifikasi
   kebakaran hutan dan gangguan alami seperti banjir yang tidak selalu kami
   tangkap dengan baik"* dan *"kebakaran sering muncul di sepanjang sungai,
   di floodplain — di tempat yang sama dengan banjir terjadi"*.
3. **Apakah error dipengaruhi musim?** (kebakaran vs. alami yang mustahil dipisahkan secara visual)

In [ ]:
# ============================================================
# 7.1 PASANGAN KELAS YANG PALING SERING TERTUKAR
# ============================================================
cm_rf = confusion_matrix(y_test, pred_rf, labels=KELAS)

pasangan = []
for i, kelas_asli in enumerate(KELAS):
    for j, kelas_pred in enumerate(KELAS):
        if i != j:
            pasangan.append({
                "Label asli": kelas_asli,
                "Prediksi": kelas_pred,
                "Jumlah": int(cm_rf[i, j]),
            })
tabel_pasangan = pd.DataFrame(pasangan).sort_values("Jumlah", ascending=False).reset_index(drop=True)

print("=== 10 PASANGAN KELAS YANG PALING SERING TERTUKAR (Random Forest) ===")
print(tabel_pasangan.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5.5))
top = tabel_pasangan.head(10).copy()
top["label_gabungan"] = top["Label asli"] + " → " + top["Prediksi"]
colors = ["#E76F51" if ("kebakaran" in l or "alami" in l) else "#457B9D" for l in top["label_gabungan"]]
ax.barh(top["label_gabungan"][::-1], top["Jumlah"][::-1], color=colors[::-1])
ax.set_title("10 Galat Tertukar Terbesar (Random Forest)\nMerah = pasangan yang melibatkan kebakaran/alami (banjir)",
             fontsize=12, weight="bold")
ax.set_xlabel("Jumlah alert yang salah diklasifikasi")
ax.set_ylabel("Label asli → Prediksi")
simpan_fig("10_pasangan_kelas_tertukar.png")

In [ ]:
# ============================================================
# 7.2 ERROR PER KELAS & DAMP AKNYA KE AKTIVITAS PENEGAKAN HUKUM
# ============================================================
baris = []
for i, kelas in enumerate(KELAS):
    support = int(cm_rf[i].sum())
    salah = int(cm_rf[i].sum() - cm_rf[i, i])
    benar = int(cm_rf[i, i])
    baris.append({
        "Kelas": kelas,
        "Support (data uji)": support,
        "Benar": benar,
        "Salah": salah,
        "Error rate": salah / support if support else np.nan,
    })
tabel_error = pd.DataFrame(baris).set_index("Kelas")
print("=== Kadar Kesalahan per Kelas (Random Forest) ===")
print(tabel_error.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 4.5))
warna = ["#E76F51" if v > 0.2 else "#2A9D8F" for v in tabel_error["Error rate"]]
ax.bar(tabel_error.index, tabel_error["Error rate"], color=warna)
ax.axhline(0.2, ls="--", color="gray", lw=1)
ax.set_title("Kadar Kesalahan per Kelas Penyebab (Random Forest)", fontsize=13, weight="bold")
ax.set_xlabel("Kelas penyebab")
ax.set_ylabel("Error rate")
ax.tick_params(axis="x", rotation=25)
simpan_fig("11_error_rate_per_kelas.png")

In [ ]:
# ============================================================
# 7.3 FOKUS: KEBAKARAN vs ALAMI (TEMUAN PEMATERI)
# ============================================================
i_keb, i_ala = KELAS.index("kebakaran"), KELAS.index("alami")

print("=== KEBAKARAN vs ALAMI (Banjir) ===")
print(f"Kebakaran salah dikira alami  : {cm_rf[i_keb, i_ala]} alert "
      f"({cm_rf[i_keb, i_ala]/cm_rf[i_keb].sum()*100:.1f}% dari alert kebakaran)")
print(f"Alami salah dikira kebakaran: {cm_rf[i_ala, i_keb]} alert "
      f"({cm_rf[i_ala, i_keb]/cm_rf[i_ala].sum()*100:.1f}% dari alert alami)")
print(f"Total galat dua arah        : {cm_rf[i_keb, i_ala] + cm_rf[i_ala, i_keb]} alert")

# Contoh konkret alert kebakaran yang salah dikira alami
df_uji = X_test.copy()
df_uji["label_asli"] = y_test.values
df_uji["prediksi_rf"] = pred_rf
df_uji["benar"] = df_uji["label_asli"] == df_uji["prediksi_rf"]

galat_keb = df_uji[(df_uji["label_asli"] == "kebakaran") & (df_uji["prediksi_rf"] == "alami")]
print(f"\nContoh {min(6, len(galat_keb))} alert kebakaran yang salah dikira 'alami':")
kolom_tampil = ["luas_ha", "ada_titik_api", "jarak_ke_sungai_km", "bulan", "jarak_ke_jalan_km"]
if len(galat_keb) > 0:
    print(galat_keb[kolom_tampil].head(6).to_string())
else:
    print("(tidak ada)")

# Seberapa sering kedua kelas ini muncul di sekitar sungai?
dekat_sungai = df_uji["jarak_ke_sungai_km"] < 3
print(f"\nAlert dengan jarak ke sungai < 3 km : {dekat_sungai.sum()} dari {len(df_uji)} "
      f"({dekat_sungai.mean()*100:.1f}%)")
print("Median jarak ke sungai (median dipakai karena rerata tergeser oleh noise label):")
for k in ["alami", "kebakaran", "tambang", "pertanian_kecil"]:
    med = df_uji[df_uji["label_asli"] == k]["jarak_ke_sungai_km"].median()
    print(f"  kelas '{k:<16s}' : {med:.2f} km")

In [ ]:
# ============================================================
# 7.4 PENGARUH MUSIM TERHADAP KESALAHAN
# ============================================================
df_uji["musim"] = np.where(df_uji["bulan"].isin(MUSIM_KEMARAU), "kemarau (Jun-Sep)", "hujan (lainnya)")

tabel_musim = (
    df_uji.groupby(["musim", "label_asli"])["benar"]
    .agg(["size", "mean"])
    .rename(columns={"size": "n", "mean": "akurasi"})
    .reset_index()
)
print("=== Akurasi menurut musim dan kelas (Random Forest) ===")
print(tabel_musim.pivot(index="label_asli", columns="musim", values="akurasi").round(3).to_string())

fig, ax = plt.subplots(figsize=(11, 5))
for musim, grup in tabel_musim.groupby("musim"):
    grup = grup.sort_values("label_asli")
    ax.plot(grup["label_asli"], grup["akurasi"], marker="o", linewidth=2,
            label=musim, color="#E76F51" if "kemarau" in musim else "#457B9D")
ax.set_title("Akurasi Klasifikasi menurut Musim (Random Forest)", fontsize=13, weight="bold")
ax.set_xlabel("Kelas penyebab")
ax.set_ylabel("Akurasi (recall per kelas)")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=25)
ax.legend()
simpan_fig("12_akurasi_musim.png")

## 8. Simulasi Operasional: Pembaruan Bulanan & Latensi

Pemateri menekankan limitation penting: *"data ini diperbarui **hanya sekali
sebulan**"* — sehingga ada **latensi** sebelum klasifikasi penyebab tersedia
untuk tindakan lapangan (mis. penargetan atau penegakan hukum).

Untuk menguji seberapa robust model terhadap **data yang "bergeser"** (konsep
*concept drift*: pola di bulan berbeda bisa berbeda dari bulan pelatihan),
kita melakukan simulasi operasional:

- Latih model dengan data **bulan 1–9** (Jan–Sep).
- Uji model tersebut pada **bulan 10, 11, 12** satu per satu (backtest bulanan).
- Bandingkan dengan akurasi pada split acak 80/20.

Interpretasi: bila akurasi pada data "mendatang" jauh turun dibanding split
acak, berarti model **tidak boleh** di-deploy tanpa monitoring & retraining
berkala — persis kebutuhan sistem peringatan yang diperbarui bulanan.

In [ ]:
# ============================================================
# 8. SIMULASI PEMBARUAN BULANAN (backtest per bulan)
# ============================================================
bulan_latih = list(range(1, 10))    # Jan-Sep
bulan_uji = [10, 11, 12]            # Okt-Des

df_latih = df[df["bulan"].isin(bulan_latih)]

# Model yang sama persis dengan Random Forest pada Bagian 4
rf_bulanan = RandomForestClassifier(
    n_estimators=400, min_samples_leaf=2,
    class_weight="balanced_subsample", random_state=SEED, n_jobs=-1
)
rf_bulanan.fit(df_latih[FITUR], df_latih[TARGET])

hasil_bulanan = []
for bln in bulan_uji:
    subset = df[df["bulan"] == bln]
    pred = rf_bulanan.predict(subset[FITUR])
    hasil_bulanan.append({
        "Bulan": f"{bln:02d}",
        "n_alert": len(subset),
        "Akurasi": accuracy_score(subset[TARGET], pred),
        "Macro F1": f1_score(subset[TARGET], pred, average="macro", zero_division=0),
    })
tabel_bulanan = pd.DataFrame(hasil_bulanan)

print("=== Backtest bulanan: latih Jan-Sep, uji Okt-Des ===")
print(tabel_bulanan.round(3).to_string(index=False))
print(f"\nRata-rata akurasi backtest bulanan : {tabel_bulanan['Akurasi'].mean():.4f}")
print(f" Akurasi split acak 80/20 (DT/RF)  : "
      f"{accuracy_score(y_test, pred_dt):.4f} / {accuracy_score(y_test, pred_rf):.4f}")
print(f" Selisih (RF backtest vs RF acak)  : "
      f"{tabel_bulanan['Akurasi'].mean() - accuracy_score(y_test, pred_rf):+.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(tabel_bulanan["Bulan"], tabel_bulanan["Akurasi"], color="#2A9D8F", width=0.55)
ax.axhline(accuracy_score(y_test, pred_rf), ls="--", color="#E76F51",
           label=f"RF split acak 80/20 = {accuracy_score(y_test, pred_rf):.3f}")
ax.set_ylim(0, 1.0)
ax.set_title("Backtest Akurasi Bulanan (Model Dilatih Jan-Sep)", fontsize=13, weight="bold")
ax.set_xlabel("Bulan pengujian")
ax.set_ylabel("Akurasi")
ax.legend()
simpan_fig("13_backtest_bulanan.png")

## 9. Kerangka Laporan

Bagian berikut adalah **kerangka laporan** yang sudah diisi dengan angka hasil
eksekusi notebook ini (seed 42). Sel kode sebelumnya juga mencetak ringkasan
angka yang sama, sehingga mudah dicocokkan saat notebook dijalankan ulang.

> Nilai-nilai di bawah **berasal dari data simulasi**, sehingga maknanya adalah
> *metodologi demonstrasi*, bukan temuan empiris terhadap hutan tropis nyata.

In [ ]:
# ============================================================
# RANGKUMAN ANGKA UNTUK LAPORAN (dibaca di sel markdown berikutnya)
# ============================================================
HASIL = {
    "n_baris": len(df),
    "n_train": len(X_train),
    "n_test": len(X_test),
    "akurasi_rule": accuracy_score(y_test, pred_rule_test),
    "akurasi_dt": accuracy_score(y_test, pred_dt),
    "akurasi_rf": accuracy_score(y_test, pred_rf),
    "f1_rule": f1_score(y_test, pred_rule_test, average="macro"),
    "f1_dt": f1_score(y_test, pred_dt, average="macro"),
    "f1_rf": f1_score(y_test, pred_rf, average="macro"),
    "cv_dt": skor_dt.mean(),
    "cv_rf": skor_rf.mean(),
    "recall_keb_rf": recall_score(y_test, pred_rf, labels=["kebakaran"], average="macro"),
    "recall_ala_rf": recall_score(y_test, pred_rf, labels=["alami"], average="macro"),
    "galat_keb_ala": int(cm_rf[i_keb, i_ala]),
    "galat_ala_keb": int(cm_rf[i_ala, i_keb]),
    "akurasi_bulanan": tabel_bulanan["Akurasi"].mean(),
    "fitur_teratas": list(perm_series.index[:3]),
}

print("=== RINGKASAN HASIL UNTUK LAPORAN ===")
for k, v in HASIL.items():
    if isinstance(v, float):
        print(f"{k:22s} : {v:.4f}")
    else:
        print(f"{k:22s} : {v}")

# Simpan ringkasan metrik sebagai CSV (berguna untuk lampiran laporan)
tabel_perbandingan.to_csv(os.path.join(DATA_DIR, "perbandingan_model.csv"))
tabel_error.to_csv(os.path.join(DATA_DIR, "analisis_kesalahan_per_kelas.csv"))
tabel_pasangan.head(10).to_csv(os.path.join(DATA_DIR, "pasangan_kelas_tertukar.csv"), index=False)
print(f"\nTabel pendukung tersimpan di: {DATA_DIR}")

### 9.1 Latar Belakang

Peringatan deforestasi (*deforestation alerts*) dihasilkan dari citra satelit
yang melintasi hutan tropis secara rutin — Sentinel-1 (radar) dan Sentinel-2
(optik) dari European Space Agency, ditambah sistem seperti GLAD, RADD, dan
TreeCoverer. Dengan pipelines pemrosesan citra, sistem ini dapat mendeteksi
**kehilangan tutupan pohon** dengan skala piksel 10–30 meter dan frekuensi
sekali per minggu.

Namun, alert hanya menjawab pertanyaan **"di mana"**, bukan **"mengapa"**.
Materi webinar yang dibawakan **Dr. Bart Slagter** (Wageningen University &
Research) menjelaskan bahwa pemecahan masalah "mengapa" ini diselesaikan dengan
**klasifikasi penyebab langsung (*direct driver*)** menggunakan *convolutional
neural network* yang membaca citra Sentinel-1 dan Sentinel-2 sebagai *stack* (tumpukan citra).
Kelas penyebab yang diklasifikasikan meliputi: pertanian skala kecil, pertanian
skala besar, penebangan selektif, pembangunan jalan, pertambangan, kebakaran
hutan, dan gangguan alami (banjir/angin).

Studi kasus ini **mengadopsi** pendekatan tersebut pada level yang
dapat dijalankan oleh mahasiswa: menguji apa yang terjadi bila
permasalahan ini diselesaikan dengan pendekatan **konvensional** — yaitu
**rule-based**, **Decision Tree**, dan **Random Forest** — di atas fitur
tabular turunan alert.

> **Catatan:** data yang digunakan adalah **data simulasi** (1.500 alert
> sintetis, seed tetap 42), sehingga hasilnya menggambarkan **alur metodologi
> dan karakteristik kesalahan model**.

### 9.2 Tujuan

1. **Membangun** dataset simulasi peringatan deforestasi yang realistis
   (1.500 baris, 7 kelas penyebab, noise label 12%) dan bersifat reproducible.
2. **Mendesain** baseline klasifikasi berbasis aturan (rule-based) sebagai
   *benchmark* yang bisa ditafsirkan manusia.
3. **Membandingkan** rule-based, Decision Tree, dan Random Forest menggunakan
   accuracy, precision, recall, F1, serta confusion matrix.
4. **Menganalisis** di mana model gagal dan mengaitkannya dengan temuan webinar
   (khususnya konfusi antara *kebakaran* dan *gangguan alami/banjir*).
5. **Menguji** keandalan model dalam skenario operasional pembaruan bulanan, untuk memperkirakan dampak latensi terhadap tindakan penegakan hukum.
6. **Merumuskan** diskusi: kapan *deep learning* perlu dipertimbangkan dan mengapa
   validasi lapangan tetap diperlukan.

### 9.3 Metode

**a. Desain data simulasi.** Tujuh kelas penyebab dibuat mengikuti logika
geografis-fisik yang dijelaskan pemateri:

| Kelas | Pola karakteristik yang disimulasikan |
|---|---|
| `pertanian_kecil` | luas kecil, bentuk kompak, relatif dekat jalan |
| `pertanian_besar` | luas besar, bentuk persegi, frekuensi tinggi di musim kemarau |
| `selective_logging` | sedang-kecil, sangat dekat jalan akses logging, bentuk agak memanjang |
| `jalan` | luas kecil, bentuk sangat memanjang, jarak ke jalan ≈ 0 |
| `tambang` | sangat dekat konsesi tambang, bentuk kompak |
| `kebakaran` | hotspot present, luas besar, terkonsentrasi Jun–Sep, sering dekat sungai |
| `alami` | sangat dekat sungai, terkonsentrasi musim hujan, 30% punya hotspot (sumber konfusi) |

Sebesar 12% baris diberi **noise label** ke kelas yang mirip (mis.
`kebakaran` → `alami`) untuk meniru ground truth yang ambigu dan ketidakpastian
anotasi lapangan. Derau kecil juga ditambahkan pada seluruh fitur
numerik.

**b. Praproses & split.** Tidak ada data hilang pada dataset sintetis ini.
Split 80/20 *stratified* (seed 42) menjaga proporsi kelas di data latih dan uji.

**c. Model.**
- *Rule-based*: 8 aturan if–else berurutan berbasis bentuk, kedekatan
  konsesi, titik api, musim, kedekatan sungai/jalan, dan luas area.
- *Decision Tree*: `max_depth=8`, `min_samples_leaf=5`, `class_weight='balanced'`.
- *Random Forest*: `n_estimators=400`, `min_samples_leaf=2`,
  `class_weight='balanced_subsample'`.
- Validasi silang 5-fold *stratified* untuk mengukur kestabilan.

**d. Evaluasi.** Accuracy, precision/recall/F1 per kelas (macro & weighted),
confusion matrix, tabel perbandingan, feature importance (gini & permutation),
visualisasi pohon (kedalaman dibatasi 4), analisis pasangan kelas yang tertukar,
analisis error per musim, serta backtest pembaruan bulanan (latih Jan–Sep,
uji Okt–Des).

### 9.4 Hasil

*(Seluruh angka berikut diambil langsung dari output sel sebelumnya.)*

**1. Karakteristik data.** 1.500 alert, 7 kelas penyebab, 1.200 data latih /
300 data uji. Distribusi kelas cukup seimbang (10,6%–21,9% per kelas), sehingga
metrik *macro-averaged* lebih bermakna daripada *accuracy* saja. Pada data uji:
`pertanian_kecil` 65, `selective_logging` 57, `pertanian_besar` 39, `tambang` 38,
`alami` 35, `kebakaran` 34, `jalan` 32 alert.

**2. Perbandingan model (data uji, n = 300).**

| Model | Accuracy | Precision (macro) | Recall (macro) | F1 (macro) | F1 (weighted) | 5-Fold CV |
|---|---|---|---|---|---|---|
| Rule-Based (if–else) | 0,7067 | 0,7382 | 0,7235 | 0,7134 | 0,6986 | — |
| Decision Tree | 0,7533 | 0,7513 | 0,7652 | 0,7548 | 0,7519 | 0,7633 ± 0,0236 |
| Random Forest | **0,8200** | **0,8200** | **0,8276** | **0,8193** | **0,8178** | 0,8167 ± 0,0257 |

Random Forest mengungguli Decision Tree pada seluruh metrik dan berselisih
**±11,3 poin** terhadap rule-based — tetapi **tidak ada model yang mencapai
100%**, justru karena noise label 12% yang disengaja (batas atas teoritis
akurasi model ≈ 0,88 pada data ini).

**3. Temuan utama — konfusi *kebakaran* vs *alami* (banjir).**
- Alert `kebakaran` yang salah dikira `alami`: **2 alert** (5,9% dari alert kebakaran).
- Alert `alami` yang salah dikira `kebakaran`: **5 alert** (14,3% dari alert alami).
- Total **7 alert** (2,3% dari data uji) berada di pasangan kelas yang sama dengan
  temuan pemateri bahwa *"kebakaran sering muncul di sepanjang sungai, di
  floodplain — di mana banjir juga terjadi"*, sehingga model pun sulit
  memisahkannya.

**4. Recall per kelas (Random Forest).**

| Kelas | Precision | Recall | F1 | Error rate |
|---|---|---|---|---|
| `jalan` | 0,762 | **1,000** | 0,865 | **0,000** |
| `pertanian_besar` | 0,833 | 0,897 | 0,864 | 0,103 |
| `tambang` | 0,919 | 0,895 | 0,907 | 0,105 |
| `selective_logging` | 0,870 | 0,825 | 0,847 | 0,175 |
| `kebakaran` | 0,750 | 0,794 | 0,771 | 0,206 |
| `pertanian_kecil` | 0,790 | 0,754 | 0,772 | 0,246 |
| `alami` | 0,815 | **0,629** | 0,710 | **0,371** |

Kelas dengan tanda geometris tegas (`jalan`, `tambang`) hampir sempurna
diklasifikasikan, sedangkan **`alami` (banjir) adalah kelas terlemah** —
konsisten dengan keterbatasan radar Sentinel-1 yang penetrasinya ke kanopi
terbatas, sehingga sinyal air di bawah tutupan rapat sulit terlihat.

**5. Error paling sering terjadi (Random Forest).** Pasangan tertukar terbesar:
`alami` → `pertanian_kecil` (7), `pertanian_kecil` → `jalan` (5),
`alami` → `kebakaran` (5), `selective_logging` → `pertanian_kecil` (4),
`kebakaran` → `pertanian_besar` (4). Bandingkan dengan rule-based, yang
mengklasifikasikan **112 alert `selective_logging` sebagai `pertanian_kecil`**
karena aturan `tambang` diperiksa lebih dulu — contoh klasik bahwa
**urutan aturan manual sangat sensitif**.

**6. Pengaruh musim.** Recall kelas `alami` anjlok dari **0,690 (musim hujan)
menjadi 0,333 (musim kemarau)** — di musim kemarau, alert BANJIR lebih sering
dikira `kebakaran` (karena sinyal hotspot ikut muncul). Recall `kebakaran`
sendiri stabil (0,789–0,800). Ini bukti kuantitatif bahwa **musim adalah fitur
penentu**, bukan sekadar fitur tambahan.

**7. Feature importance (permutation importance, Random Forest).**
`kedekatan_tambang_km` (0,2467) > `luas_ha` (0,2237) > `jarak_ke_jalan_km`
(0,1717) > `jarak_ke_sungai_km` (0,0803) > `bentuk_memanjang` (0,0650) >
`ada_titik_api` (0,0497) > `bulan` (0,0060). Model ternyata membaca
**konteks kedekatan (mines & roads) dan ukuran** lebih dulu, bukan hanya
bentuk geometris — selaras dengan pemateri, yang menyatakan modelnya tidak
hanya melihat ukuran/bentuk alert, tetapi juga **konteks di sekitarnya**.

**8. Backtest bulanan.** Model yang dilatih Jan–Sep dan diuji Okt–Des
menghasilkan akurasi **0,803 (Okt), 0,827 (Nov), 0,832 (Des)** dengan rata-rata
**0,8207** — praktis identik dengan akurasi pada split acak (0,8200, selisih
+0,0007). Artinya model **tahan terhadap pergeseran musiman** pada data
simulasi ini; namun kesetaraan ini bergantung pada asumsi bahwa pola tidak
berubah (*concept drift*) dan pada ketersediaan citra yang stabil — dua hal
yang dalam dunia nyata tidak dapat diasumsikan, sehingga sistem peringatan
bulanan tetap memerlukan monitoring dan retraining berkala.

**9. Lampiran tabel.** Tabel metrik per kelas tersimpan di file
`data/perbandingan_model.csv`, `data/analisis_kesalahan_per_kelas.csv`, dan
`data/pasangan_kelas_tertukar.csv`.

### 9.5 Pembahasan

**a. Mengapa Random Forest menang?** Forest mengagregasi ratusan pohon dengan
*bootstrap sampling* dan *feature randomness*, sehingga lebih tahan overfit
dan lebih stabil dibanding satu pohon (selisih 5-komponen CV: 0,8167 vs
0,7633). Pohon yang dalam juga menangkap interaksi non-linier antar fitur
(mis. "dekat sungai **dan** ada hotspot **dan** bulan kemarau" → pola
drainase) yang mustahil dituliskan dengan if–else sederhana. Selain itu, model
juga belajar dari *konteks* — sesuai penjelasan pemateri.

**b. Mengapa rule-based kalah jauh (0,7067 vs 0,8200)?** Aturan manual hanya
memanfaatkan satu-satu fitur secara berurutan, tanpa dapat mengombinasikan
bukti. Akibatnya: (1) **urutan aturan** menentukan hasil — pada seluruh data,
**112 alert `selective_logging`** (yang konsesinya berdekatan dengan konsesi
tambang) salah diklasifikasikan sebagai `pertanian_kecil`, dan pada data uji
20 dari 57 alert `selective_logging` (35%) salah masuk kelas `pertanian_kecil`;
(2) recall `selective_logging` hanya **0,439** (precision 0,893) — model
sangat *konservatif* untuk kelas ini, artinya **banyak alert logging yang
tidak pernah dikenali**; (3) kelas `alami` hanya dikenali pada 54,3% alert
banjir. Aturan bisa diperbaiki manual, tetapi setiap perbaikan baru harus
diuji ulang ke seluruh data. Inilah alasan mengapa pendekatan berbasis data
lebih mudah diskalakan untuk data berskala besar.

**c. Mengapa masih ada error ±12%?** Karena error tersebut telah ditanamkan
secara sengaja pada label (noise 12%). Artinya, dengan data ini, **limit
akurasi teoretis model adalah sekitar 0,88**, dan Random Forest sudah
mencapainya (0,8200). Kesimpulan metodologisnya: bagi model klasifikasi driver
deforestation, **kualitas label lebih menentukan daripada arsitektur model**.
Meningkatkan hanya arsitektur (CNN) tanpa memperbaiki anotasi label tidak akan
menghilangkan error ini.

**d. Konfusi kebakaran vs banjir — relevansi dengan materi.** Data simulasi
meniru temuan pemateri: kebakaran (*fire*) muncul di sepanjang sungai, dan
*flooding* terjadi di lokasi yang sama, sehingga keduanya saling meniru
pada citra. Pada bagian Q&A, pemateri menambahkan bahwa **Sentinel-1
(radar) sangat sensitif terhadap air dan kelembapan**, tetapi penetrasinya ke
kanopi rapat terbatas. Konsekuensinya: untuk membedakan *fire* dan *flood*,
kombinasi **citra optik multi-temporal (Sentinel-2)** + **informasi
musiman/curah hujan** jauh lebih informatif daripada satu fitur biner
`ada_titik_api`. Hasil eksperimen ini mendukung hal yang sama secara
kuantitatif: recall `alami` di musim kemarau hanya 0,333, dan 5 dari 35 alert
banjir salah dikira kebakaran.

**e. Kapan deep learning perlu dipertimbangkan?** Sesuai jawaban pemateri,
pertimbangkan *deep learning* (CNN) bila: (1) inputnya berupa *stack* citra
multi-sensor yang kaya informasi spasial, (2) pola yang dicari bersifat
spasial-temporal (mis. jejak jalan yang memanjang bertahap, atau rangkaian
*mining → agriculture* yang ditunjukkan pemateri), (3) data berlabel sangat
banyak, atau (4) masalahnya memerlukan pembacaan konteks yang tidak dapat
dirumuskan sebagai fitur tabel. Pada studi kasus ini, pendekatan rule-based →
tree → forest sudah memberi peningkatan yang bermakna, sesuai saran pemateri
untuk selalu memulai dari *benchmark* konvensional. **Kesimpulannya bukan
"deep learning tidak berguna", melainkan "deep learning harus dibayar dengan
data berlabel mahal"** — CNN hanya unggul bila data dan konteks spasialnya
memang tersedia.

**f. Dampak operasional untuk penegakan hukum.** Presisi dan recall
sama-sama penting: presisi rendah berarti *false alarm* yang menyedot sumber
daya lapangan; recall rendah berarti deforestasi ilegal yang lolos radar.
Klasifikasi penyebab membantu **memprioritaskan** alert (mis.
*selective_logging/jalan* = membuka akses, *tambang* = illegalitas jelas)
sehingga sumber daya diarahkan ke kasus berdampak tertinggi. Justru kelas
`alami` yang paling lemah recall-nya adalah kandidat prioritas untuk
perbaikan: alert banjir yang salah dikira kebakaran justru berisiko
memicu tindakan hukum yang tidak berdasar.

**g. Keterbatasan data simulasi.** (1) Ground truth sintetis "terlalu bersih"
sebelum noise diterapkan — data lapangan jauh lebih berantakan. (2) Hanya 7
fitur tabuler; citra Sentinel-1/2 yang kaya informasi spasial tidak
diwakili. (3) Tidak ada variabel geografis (wilayah, negara, vegetasi,
kepemilikan lahan, konservasi) padahal pemateri menunjukkan perbedaan kuat
antar wilayah (Amazon vs Basin Congo vs Asia Tenggara). (4) Musim
menggunakan pola global sederhana, bukan pola per wilayah. (5) Tidak ada
ketidakseimbangan temporal (update bulanan) maupun *concept drift* yang
sebenarnya. (6) Biaya anotasi label yang tinggi tidak tergantikan di dalam
data simulasi ini. (7) **Tanpa validasi lapangan, tidak ada satu pun angka
dalam laporan ini yang boleh dipakai sebagai estimasi kinerja di dunia nyata.**

### 9.6 Kesimpulan

1. **Rule-based tetap berguna sebagai baseline pertama** (accuracy 0,7067) untuk
   menyaring alert kasar — ini mendukung pesan pemateri bahwa metode
   konvensional harus dicoba lebih dulu sebelum propos model yang rumit.
   Namun ia gagal pada kasus tumpang-tindih: recall `selective_logging` hanya
   0,439 dan `alami` hanya 0,543, dengan 112 alert logging yang salah
   diklasifikasikan karena urutan aturan.
2. **Random Forest adalah pendekatan terbaik di antara tiga metode**
   (accuracy 0,8200; macro-F1 0,8193; 5-fold CV 0,8167 ± 0,0257), konsisten
   mengungguli Decision Tree (0,7533) dan rule-based (0,7067).
3. **Tidak ada model yang mencapai akurasi sempurna**, dan itu memang
   seharusnya: label penyebab deforestasi secara mendasar bersifat ambigu di
   lapangan. **Kualitas label, bukan kompleksitas model, adalah pembatas
   utama** — pesan yang selaras dengan presentasi pemateri.
4. **Kesalahan model terkonsentrasi pada pasangan kelas yang memang ambigu
   secara fisis**: *kebakaran* ↔ *alami/banjir* (7 dari 300 alert uji), dengan
   `alami` sebagai kelas terlemah (recall 0,629; error rate 0,371) dan recall
   yang anjlok dari 0,690 (musim hujan) menjadi 0,333 (musim kemarau).
   Perbaikan harus datang dari sumber data tambahan (citra optik
   multi-temporal, data curah hujan dan hotspot historis, validasi lapangan),
   bukan sekadar menambah kompleksitas model.
5. **Performa model stabil terhadap pergeseran musiman pada data simulasi**
   (backtest bulanan 0,8207 vs split acak 0,8200) — namun stabilitas ini
   bergantung pada asumsi tidak adanya *concept drift* dan kegagalan satelit,
   sehingga sistem peringatan bulanan tetap memerlukan monitoring dan
   retraining berkala (materi: latensi pembaruan bulanan).
6. **Deep learning (CNN multi-sensor) layak dipertimbangkan pada tahap
   berikutnya** — terutama dengan data berlabel yang jauh lebih banyak, input
   citra Sentinel-1 + Sentinel-2, dan kebutuhan membaca konteks spasial-temporal
   (mis. urutan *mining → agriculture*, *logging → fire* pada contoh webinar).
   Namun hasilnya harus **divalidasi dengan data lapangan**, karena pada
   dasarnya citra satelit hanya menunjukkan *jejak* dan *konteks*, bukan
   sebab yang sebenarnya.

## 10. Referensi & Catatan Penutup

**Materi utama**
- Webinar *"Deforestation Alerts and Driver Classification"* — Dr. Bart Slagter
  (Wageningen University & Research), Lazy Bear Watch Team.
- Platform rujukan yang disebutkan pemateri: **Global Forest Watch /
  Global Nature Watch** (WRI), **MapBiomas** & **MapBiomas Alerta** (Brasil),
  **GLAD Alerts** (University of Maryland), Sentinel-1 & Sentinel-2 (ESA).

**Pustaka Python yang digunakan**
`pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`.

**Cara menjalankan**
1. Google Colab: `File → Import notebook` → pilih file `notebooks/studi_kasus_deforestation_drivers.ipynb` → `Run all`.
2. Lokal: `pip install -r ../requirements.txt` lalu jalankan sel demi sel di Jupyter.

**File keluaran yang dihasilkan**
- `data/alert_deforestation_simulasi.csv` — dataset simulasi 1.500 baris.
- `data/perbandingan_model.csv` — tabel perbandingan metrik model.
- `data/analisis_kesalahan_per_kelas.csv` — error rate per kelas.
- `data/pasangan_kelas_tertukar.csv` — 10 pasangan tertukar terbesar.
- `data/figures/` — seluruh grafik (PNG).

> **Penutup:** seluruh angka pada laporan ini berasal dari **data simulasi**.
> Coin Ini adalah bahan ajar untuk memahami alur *geospatial machine
> learning*, bukan estimasi kinerja model di dunia nyata.